# Experimento: MLP, fase 7 (as features do FT e Optuna com os dados)

**De onde vem.** No FT-Transformer, a análise de erro gerou features novas. A razão de cobrança `cobranca_razao` = TotalCharges / (MonthlyCharges × tenure) foi a que mais subiu o KS OOF (+0,0047). O pacote das cinco com ganho levou o FT ao maior KS do placar dele (0,5573), e o Optuna do FT preferiu `cobranca_razao` + `antigo` em todos os melhores trials. **Nada disso foi medido no MLP.** O Optuna do MLP (fase 5) é anterior a essas features.

## 1. Triagem nas divisões próprias

Sobre o MLP adotado (`s6_rmsprop_ks_p50`: 1×10, rmsprop, parada pelo KS com paciência 50), nas **sementes 201 a 203** e não nas do placar:

| Candidato | O que acrescenta |
|---|---|
| `m_cobranca_razao` | TotalCharges / (MonthlyCharges × max(tenure, 1)) |
| `m_antigo` | mais de 48 meses de casa |
| `m_caro_sem_backup` | mensalidade > 98 e sem backup |
| `m_fibra_sem_protecao` | fibra sem nenhum serviço de proteção |
| `m_atrito_pagamento` | mês a mês, cheque eletrônico e fatura digital |
| `p_razao_antigo` | `cobranca_razao` + `antigo` |
| `p_cinco` | as cinco |

## 2. Optuna com o conjunto de dados como eixo

O Optuna escolhe entre a base e os dois melhores candidatos da triagem, junto com a arquitetura (1 ou 2 camadas, 4 a 32 unidades), a ativação, o otimizador, a taxa de aprendizado, o L2, o dropout, o lote, o oversampling e a regra de parada. Os primeiros trials são o MLP adotado com cada conjunto. Cada trial vale o KS médio de 15 dobras próprias. Os trials treinam em lotes de 8 na pilha da GPU, e o estudo fica em sqlite: retomar a execução retoma o estudo.

## 3. Placar

O melhor da triagem com os hiperparâmetros do adotado (`m7_triagem`) e os 3 melhores trials (`m7_top1` a `m7_top3`) vão ao placar, contra o trial 0 e contra o MLP adotado (`vs_base`).

## Como executar no Kaggle

1. *File → Import Notebook* → `experimentos/mlp/mlp_fase7.ipynb`, com o dataset anexado.
2. *Add-ons → Secrets:* `GITHUB_TOKEN` e `WANDB_API_KEY`. **Nunca escreva os tokens neste notebook.**
3. *Settings:* Internet **On** e acelerador **GPU T4 x2**.
4. Rode com `MODO_TESTE = True`. Depois, com `False`, use *Save Version → Save & Run All (Commit)*. Para continuar uma execução que caiu, use `RUN_NAME` com o mesmo nome e `RESUME_FROM = "github"`.

# 0. Preparação do ambiente

In [ ]:
# ===== Configuração da execução (edite aqui) =====
REPO_URL = "https://github.com/diegoflyra/churn_project.git"
BRANCH = "diego/ft-transformer"              # branch com o código (a fase 7 está nela)
RESULTS_REPO_URL = REPO_URL      # onde espelhar os resultados ("" = não enviar ao GitHub)
RESULTS_BRANCH = "resultados"    # branch órfã só de resultados
RUN_NAME = ""                    # vazio = "mlp-fase7-AAAAMMDD-HHMM"; fixe um nome para retomar
RESUME_FROM = ""                 # "" | "github" | pasta com os resultados de uma execução anterior
CSV_PATH = "/kaggle/input/datasets/diegolyra/churn-ibm/telco_ibm.csv"                    # caminho do CSV; vazio = procurar no repositório e em /kaggle/input
USAR_GPU = True                  # placar em pilha na(s) GPU(s); sem GPU, uma rede por vez em vários processos
DISPOSITIVO = "cpu"              # onde roda modelo.py sem GPU: uma rede pequena por vez rende mais na CPU
N_PROCESSOS = 0                  # 0 = todos os núcleos
N_TRIALS = 120                   # trials do Optuna (cada um mede 15 dobras próprias)
TOP_OPTUNA = 3                   # melhores trials que vão para o placar
USAR_WANDB = True                # False = não envia nada ao W&B, mesmo com a chave
USAR_GITHUB = True               # False = não espelha os resultados no GitHub, mesmo com o token
MODO_TESTE = False               # True = passada rápida para validar o notebook; os números não servem como resultado
HORAS_LIMITE = 10.5              # depois disso não começa etapas novas e encerra limpo (o Kaggle corta em 12 h)

In [ ]:
import importlib.util, os, shutil, subprocess, sys, time
from pathlib import Path

INICIO = time.time()


def _load_dotenv(path):
    """Tokens em um .env local (GITHUB_TOKEN, WANDB_API_KEY); nunca sobrescreve variáveis já definidas."""
    if os.path.isfile(path):
        for line in open(path, encoding="utf-8"):
            key, sep, value = line.strip().partition("=")
            if sep and not key.startswith("#") and value.strip() and not os.environ.get(key.strip()):
                os.environ[key.strip()] = value.strip().strip("'").strip('"')


def _secret(name):
    value = os.environ.get(name, "").strip()
    if value:
        return value
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        pass
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        pass
    return ""


def _is_repo(p):
    return (p / "comum.py").exists() and (p / "experimentos" / "mlp" / "modelo.py").exists()


local = next((p for p in [Path.cwd(), *Path.cwd().parents] if _is_repo(p)), None)
if local:
    REPO_DIR = local
    _load_dotenv(REPO_DIR / ".env")
    GITHUB_TOKEN = _secret("GITHUB_TOKEN")
    print(f"Código: cópia local em {REPO_DIR}")
else:
    GITHUB_TOKEN = _secret("GITHUB_TOKEN")
    REPO_DIR = Path("/tmp/churn_project")
    url = REPO_URL.replace("https://", f"https://x-access-token:{GITHUB_TOKEN}@", 1) if GITHUB_TOKEN else REPO_URL
    print("GitHub: usando GITHUB_TOKEN" if GITHUB_TOKEN else "GitHub: clone sem token")
    shutil.rmtree(REPO_DIR, ignore_errors=True)
    if subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, url, str(REPO_DIR)]).returncode != 0:
        raise RuntimeError(f"Falha ao clonar a branch {BRANCH}. Confira se ela foi publicada; repositório privado exige GITHUB_TOKEN.")
    subprocess.run(["git", "-C", str(REPO_DIR), "remote", "set-url", "origin", REPO_URL])  # o token não fica salvo no clone
    print(f"Código: {REPO_DIR}")
os.chdir(REPO_DIR)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout.strip())

# Instala só o que falta: o Kaggle já traz versões compatíveis com a GPU dele
IMPORTS = {"imbalanced-learn": "imblearn", "scikit-learn": "sklearn"}
faltam = [p.strip() for p in open("requirements.txt", encoding="utf-8")
          if p.strip() and importlib.util.find_spec(IMPORTS.get(p.strip(), p.strip())) is None]
if faltam:
    print("Instalando:", ", ".join(faltam))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *faltam])

In [ ]:
import json

sys.path[:0] = [str(REPO_DIR), str(REPO_DIR / "experimentos" / "mlp")]
os.environ["PYTHONPATH"] = os.pathsep.join([str(REPO_DIR), str(REPO_DIR / "experimentos" / "mlp"), os.environ.get("PYTHONPATH", "")])  # processos paralelos acham os módulos
os.environ.setdefault("TORCH_DISABLE_NATIVE_JIT", "1")
import fluxo

WORKDIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content" if os.path.isdir("/content") else str(REPO_DIR)
RUN_NAME = RUN_NAME or time.strftime("mlp-fase7-%Y%m%d-%H%M")
if MODO_TESTE and not RUN_NAME.startswith("teste-"):
    RUN_NAME = "teste-" + RUN_NAME
# O teste nunca se mistura com a execução real, e rodar localmente não sobrescreve o que está versionado em resultados/
PASTA = os.path.join(WORKDIR, "execucoes", RUN_NAME) if WORKDIR == str(REPO_DIR) else os.path.join(WORKDIR, "resultados_" + RUN_NAME)
os.makedirs(PASTA, exist_ok=True)
os.environ["CHURN_RESULTADOS"] = PASTA
os.environ["CHURN_CSV"] = fluxo.achar_csv(CSV_PATH)
if DISPOSITIVO:
    os.environ["CHURN_DEVICE"] = DISPOSITIVO
N_PROCESSOS = N_PROCESSOS or os.cpu_count() or 1
if MODO_TESTE:
    os.environ["CHURN_MAX_EPOCAS"] = "3"
    N_TRIALS, TOP_OPTUNA = 4, 1
    print("MODO_TESTE: 3 épocas, dois candidatos e 4 trials. Os números não servem como resultado.")

if not USAR_GITHUB:
    RESULTS_REPO_URL = ""
if RESUME_FROM == "github":
    fluxo.baixar_github(PASTA, RUN_NAME, RESULTS_REPO_URL, GITHUB_TOKEN, RESULTS_BRANCH)
elif RESUME_FROM:
    shutil.copytree(RESUME_FROM, PASTA, dirs_exist_ok=True)


def salvar():
    return fluxo.salvar_github(PASTA, RUN_NAME, RESULTS_REPO_URL, GITHUB_TOKEN, RESULTS_BRANCH)


def salvar_json(nome, obj):
    json.dump(obj, open(os.path.join(PASTA, nome), "w"), indent=1, default=float)
    salvar()


_ultimo_backup = [time.time()]


def backup_periodico():
    if time.time() - _ultimo_backup[0] > 30 * 60:
        salvar(); _ultimo_backup[0] = time.time()


def continuar():
    return HORAS_LIMITE * 3600 - (time.time() - INICIO) > 0


import matplotlib.pyplot as plt, numpy as np, pandas as pd
from scipy.stats import ks_2samp
import comum, modelo as M, fase4 as F4, fase5 as F5, fase7 as F7, pilha as P
import torch

GPUS = P.dispositivos_disponiveis() if USAR_GPU and torch.cuda.is_available() else None
F5.usar_pilha(GPUS)
pd.set_option("display.max_columns", None, "display.width", 220, "display.max_colwidth", 70, "display.max_rows", 80)
print(f"RUN_NAME = {RUN_NAME} | resultados em {PASTA}")
print(f"CSV: {os.environ['CHURN_CSV']} | pilha: {GPUS or 'desligada'} | modelo.py: {M.DEVICE} | processos: {N_PROCESSOS}")

# O envio ao GitHub falhou a fase 3 inteira sem ninguém perceber: aqui ele é testado antes de qualquer treino
json.dump(dict(run=RUN_NAME, inicio=time.strftime("%Y-%m-%d %H:%M")), open(os.path.join(PASTA, "inicio.json"), "w"))
GITHUB_OK = bool(RESULTS_REPO_URL and GITHUB_TOKEN) and salvar()
if RESULTS_REPO_URL and not GITHUB_OK:
    print("=" * 90 + "\nATENÇÃO: o envio ao GitHub falhou. Troque o GITHUB_TOKEN nos Secrets ou baixe a pasta de resultados no fim.\n" + "=" * 90)
else:
    print("GitHub:", "envio testado e funcionando" if GITHUB_OK else "desligado")

wandb_run = None
wandb_key = _secret("WANDB_API_KEY") if USAR_WANDB else ""
if wandb_key or os.environ.get("WANDB_MODE") == "offline":
    import wandb
    if wandb_key:
        os.environ["WANDB_API_KEY"] = wandb_key
    wandb_run = wandb.init(project="churn-if702", entity=_secret("WANDB_ENTITY") or None, name=RUN_NAME, dir=WORKDIR,
                           group="mlp", config=dict(modelo="mlp", fase=7, processos=N_PROCESSOS, gpus=GPUS, teste=MODO_TESTE))
else:
    print("W&B desativado (sem chave ou USAR_WANDB = False).")


def logar(**k):
    if wandb_run is not None:
        wandb_run.log(k)


# As linhas do MLP já medidas (resultados/placar.jsonl, versionado no repositório) entram aqui sem treinar de novo
if not os.path.isfile(os.path.join(PASTA, "placar.jsonl")):
    _orig = os.path.join(REPO_DIR, "resultados")
    with open(os.path.join(PASTA, "placar.jsonl"), "w", encoding="utf-8") as _f:
        for _r in comum.ler_placar(_orig):
            if _r["modelo"] == "mlp":
                _f.write(json.dumps(_r) + "\n")
                os.makedirs(os.path.join(PASTA, "placar_oof"), exist_ok=True)
                shutil.copy(os.path.join(_orig, _r["oof"]), os.path.join(PASTA, _r["oof"]))
anteriores = {r["rotulo"]: r for r in comum.ler_placar(PASTA) if r["modelo"] == "mlp"}
print(f"Placar do MLP: {len(anteriores)} linhas trazidas de resultados/placar.jsonl")
assert F7.ROT_BASE in anteriores and F7.chave(F7.BASE) == F5._chave_cfg(anteriores[F7.ROT_BASE]["config"]), "o MLP adotado não está no placar"


In [ ]:
# Verificações rápidas antes de gastar tempo de treino
assert comum.assinatura_teste() == "e90ba6fab761f037", "o teste mudou: os números não seriam comparáveis com os outros modelos"
rng = np.random.default_rng(1); y_ = (rng.random(2000) < .265).astype(int); p_ = np.clip(rng.normal(.3 + .25 * y_, .2), 0, 1)
assert abs(comum.ks(y_, p_) - ks_2samp(p_[y_ == 1], p_[y_ == 0]).statistic) < 1e-9
D = F5.divs()
assert len(D) == 15
for d in D:
    assert not set(d["treino"]) & set(d["validacao"]) and not set(d["ajuste"]) & set(d["parada"])
y_treino = F4._dados()[2].values.astype(int)
print(f"teste com a assinatura oficial | KS conferido contra scipy | 15 divisões do placar sem sobreposição | assinatura do treino {F5.assinatura()}")


def mostrar(linhas, campos=("vs_trial0", "vs_base")):
    display(F5.tabela(linhas, campos).round(4))
    salvar()
X_, y_ = F4._dados()[0], F4._dados()[2].values
for f in F7.QUASE:
    v = comum.FEATURES[f](X_)
    print(f"{f:<20}", f"{int(v.sum()):>5} clientes, churn {y_[v == 1].mean():.3f} contra {y_[v == 0].mean():.3f}" if v.nunique() <= 2
          else f"numérica, correlação com o churn {np.corrcoef(v, y_)[0, 1]:+.3f}")

# 1. Triagem nas divisões próprias

`delta_oof` é o ganho no KS OOF contra o MLP adotado nas mesmas divisões, e `p_positivo` é a fração das reamostragens do bootstrap com ganho. `passou` = ganho em pelo menos 80% delas. Os dois de maior `delta_oof` (com ganho) entram no Optuna como opção de conjunto de dados.

In [ ]:
CANDS = F7.candidatos()
if MODO_TESTE:
    CANDS = {k: CANDS[k] for k in ("m_antigo", "p_razao_antigo")}
D_OPT = F7.divs_optuna(F7.SEMENTES_OPTUNA[:1] if MODO_TESTE else F7.SEMENTES_OPTUNA)
arq_t = os.path.join(PASTA, "triagem.json")
if os.path.isfile(arq_t):
    TRI = json.load(open(arq_t))
else:
    t0 = time.time()
    TRI = F7.triagem(CANDS, D=D_OPT, n_jobs=N_PROCESSOS)
    print(f"{(time.time() - t0) / 60:.1f} min")
    salvar_json("triagem.json", TRI)
print(f"MLP adotado nessas divisões: KS {TRI['base']['ks_kfold']:.4f} | KS OOF {TRI['base']['ks_oof']:.4f}")
tt = pd.DataFrame(TRI["linhas"]).sort_values("delta_oof", ascending=False)
tt["ic95"] = tt.ic95.map(lambda v: f"[{v[0]:+.4f}, {v[1]:+.4f}]")
display(tt.round(4))
for l in TRI["linhas"]:
    logar(**{"triagem/candidato": l["candidato"], "triagem/delta_oof": l["delta_oof"], "triagem/p_positivo": l["p_positivo"]})
arq_c = os.path.join(PASTA, "optuna_conjuntos.json")
if os.path.isfile(arq_c):                         # retomar: o eixo do Optuna não pode mudar no meio do estudo
    CONJ = json.load(open(arq_c))
else:
    CONJ = F7.conjuntos_optuna(TRI["linhas"], CANDS)
    salvar_json("optuna_conjuntos.json", CONJ)
print("Conjuntos no Optuna:", {k: v[3:] for k, v in CONJ.items()})

# 2. Optuna (conjunto de dados, arquitetura e treino)

In [ ]:
import optuna


def ao_lote(st):
    logar(**{"optuna/trials": len(st.trials), "optuna/melhor": st.best_value})
    backup_periodico()


t0 = time.time()
estudo = F7.estudar(PASTA, N_TRIALS, CONJ, D=D_OPT, n_jobs=N_PROCESSOS, continuar=continuar, ao_lote=ao_lote)
print(f"{(time.time() - t0) / 60:.1f} min | {len(estudo.trials)} trials")
df_opt = estudo.trials_dataframe().sort_values("value", ascending=False)
df_opt.to_csv(os.path.join(PASTA, "optuna_trials.csv"), index=False)
display(df_opt.head(10).round(4))
salvar_json("optuna_resumo.json", F5.resumo_optuna(estudo))
try:
    imp = optuna.importance.get_param_importances(estudo)
    print("Importância dos eixos (fANOVA):"); display(pd.Series(imp).round(3).to_frame("importância"))
    salvar_json("optuna_importancia.json", imp)
except Exception as e:
    print("importância indisponível:", e)
salvar()

# 3. No placar

`vs_base` é o veredito contra o MLP adotado (`s6_rmsprop_ks_p50`), nas dobras do placar.

In [ ]:
T0, B = ("mlp", "trial0"), ("mlp", F7.ROT_BASE)
contra = {"vs_trial0": T0, "vs_base": B}
itens = []
bons = sorted([l for l in TRI["linhas"] if l["delta_oof"] > 0], key=lambda l: -l["delta_oof"])
if bons:
    itens.append(("m7_triagem", CANDS[bons[0]["candidato"]], contra))
    print("m7_triagem =", bons[0]["candidato"])
excluir = {F7.chave(F7.BASE)} | {F7.chave(c) for _, c, _ in itens}
tops = F7.melhores(estudo, CONJ, TOP_OPTUNA, excluir)
itens += [(f"m7_top{i + 1}", F7.config_do_trial(t.params, CONJ), contra) for i, t in enumerate(tops)]
for t, (r, c, _) in zip(tops, itens[-len(tops):] if tops else []):
    print(r, f"| trial {t.number} | KS Optuna {t.value:.4f} |", json.dumps(c, sort_keys=True))
t0 = time.time()
linhas = F5.garantir(PASTA, itens, n_jobs=N_PROCESSOS)
print(f"{(time.time() - t0) / 60:.1f} min")
for r in linhas:
    v = r.get("vs_base") or {}
    logar(rotulo=r["rotulo"], ks_placar=r["ks_kfold"], ep=r["ep"], ks_oof=r["ks_oof"], auc_oof=r["auc_oof"], epocas=r.get("epocas"),
          delta_vs_base=v.get("delta"))
mostrar(linhas)

# 4. Placar do MLP

In [ ]:
todas = [r for r in comum.ler_placar(PASTA) if r["modelo"] == "mlp"]
tab = F5.tabela(sorted(todas, key=lambda r: -r["ks_kfold"]), ("vs_trial0", "vs_base"))
display(tab.round(4))
tab.to_csv(os.path.join(PASTA, "placar_mlp_fase7.csv"), index=False)
open(os.path.join(PASTA, "PLACAR.md"), "w", encoding="utf-8").write(comum.relatorio_placar(PASTA))
ganharam = [r["rotulo"] for r in todas if (r.get("vs_base") or {}).get("veredito") == "ganhou"]
perderam = [r["rotulo"] for r in todas if (r.get("vs_base") or {}).get("veredito") == "perdeu"]
print(f"Ganharam do MLP adotado: {ganharam or 'nenhuma'}\nPerderam: {perderam or 'nenhuma'}")
salvar()
if wandb_run is not None:
    wandb_run.summary.update(dict(ganharam=ganharam, perderam=perderam))
    wandb_run.finish()
print(f"Tempo total: {(time.time() - INICIO) / 3600:.2f} h")
print("Resultados em", PASTA)
print(sorted(os.listdir(PASTA)))

### 📝 Análise da fase 7

- **Alguma feature ou pacote do FT ajudou o MLP na triagem?** _…_
- **Qual conjunto e qual arquitetura o Optuna preferiu?** _…_
- **Algum `m7_*` ganhou do MLP adotado no placar?** _…_
- **Muda o MLP adotado?** Pela regra, o maior KS do placar. O teste continua usado uma vez, na configuração final. _…_